# Практическая работа по алгоритму K-Means Clustering
Алгоритм машинного обучения без учителя (Unsupervised Learning). Значения целевой переменной (таргета) неизвестны.

Цель алгоритма - минимизировать суммарное квадратичное отклонение точек кластера от их центров (центроидов).

### Алгоритм:
1. **Инициализация**: Выбирается число $k$ (количество кластеров) и случайным образом расставляются $k$ начальных центроидов в пространстве данных.
2. **Распределение по кластерам**: Каждая точка данных относится к тому кластеру, чей центроид находится к ней ближе всего (обычно используется евклидово расстояние).
    - Евклидово расстояние - это самый прямой путь между двумя точками в пространстве. $d(p ,q) = \sqrt{ \sum_{i=1}^n (p_i - q_i)^2 }$
3. **Обновление центроидов**: Для каждого полученного кластера вычисляется новый центр масс — среднее арифметическое координат всех точек, вошедших в этот кластер.
4. **Повторение**: Шаги 2 и 3 повторяются до тех пор, пока центроиды не перестанут заметно перемещаться или не будет достигнуто максимальное число итераций.

### Функция стоимости (Inertia):
В K-means функцией стоимости является сумма квадратов внутрикластерных расстояний. WCSS (Within-Cluster Sum of Squares) или Инерция (Inertia).
$$J = \sum_{j=1}^k \sum_{x \in C_j} ||x - \mu_j||^2$$
- $k$ - количество кластеров.
- $C_j$ - множество точек, принадлежащих $j$-му кластеру.
- $x$ - конкретная точка данных.
- $\mu_j$ - центроид (среднее значение) $j$-го кластера.
- $||x - \mu_j||^2$ - квадрат евклидова расстояния между точкой и центроидом.


Цель работы - Реализовать алгоритм K-means с нуля и применить его для сжатия изображений.

## 1. Реализация K-Means

### 1.1 Функция `find_closest_centroids`
Эта функция находит ближайший центроид для каждой точки

Принцип работы:
1. Для каждой точки данных вычисляем расстояние до всех центроидов
2. Находим индекс центроида с минимальным расстоянием

In [ ]:
def find_closest_centroids(X, centroids):
    """
    Находит ближайший центроид для каждого примера

    Параметры:
        X: (m, n) - матрица данных (m примеров, n признаков)
        centroids: (K, n) - координаты центроидов

    Возвращает:
        idx: (m,) - индексы ближайших центроидов
    """
    K = centroids.shape[0]
    idx = np.zeros(X.shape[0], dtype=int)

    # Перебираем каждый пример данных
    for i in range(X.shape[0]):
        # Массив для хранения расстояний
        distances = []

        # Перебираем каждый центроид
        for j in range(K):
            # Вычисляем евклидово расстояние
            distance = np.linalg.norm(X[i] - centroids[j])
            distances.append(distance)

        # Находим индекс ближайшего центроида
        idx[i] = np.argmin(distances)

    return idx

### 1.2 Функция `compute_centroids`
Эта функция пересчитывает положение центроидов на основе назначенных им точек.

In [ ]:
def compute_centroids(X, idx, K):
    """
    Вычисляет новые центроиды как среднее арифметическое точек в кластере

    Параметры:
        X: (m, n) - матрица данных
        idx: (m,) - индексы кластеров для каждой точки
        K: int - количество кластеров

    Возвращает:
        centroids: (K, n) - новые координаты центроидов
    """
    m, n = X.shape
    centroids = np.zeros((K, n))

    # Перебираем каждый кластер
    for k in range(K):
        # Выбираем все точки, принадлежащие кластеру k
        points = X[idx == k]

        # Вычисляем среднее арифметическое (новый центроид)
        if len(points) > 0:  # избегаем деления на ноль
            centroids[k] = np.mean(points, axis=0)

    return centroids

## 2. Основной алгоритм K-Means

Алгоритм:
1. Инициализация центроидов
2. Для каждой итерации:
    - Назначить точки ближайшим центроидам (cluster assignment)
    - Пересчитать центроиды (move centroid)

In [ ]:
def run_kMeans(X, initial_centroids, max_iters=10, plot_progress=False):
    """
    Запускает алгоритм K-means
    Сходимость: алгоритм гарантированно сходится к локальному оптимуму
    """
    m, n = X.shape
    K = initial_centroids.shape[0]
    centroids = initial_centroids
    idx = np.zeros(m)

    for i in range(max_iters):
        # Шаг 1: Назначение точек кластерам
        idx = find_closest_centroids(X, centroids)

        # Шаг 2: Пересчет центроидов
        centroids = compute_centroids(X, idx, K)

        print(f"Итерация {i+1}/{max_iters} завершена")

    return centroids, idx

## 3. Инициализация центроидов

Случайная инициализация центроидов. Почему случайная? Разные инициализации могут привести к разным результатам (локальные оптимумы)


In [ ]:
def kMeans_init_centroids(X, K):
    """
    Случайная инициализация центроидов
    """
    # Перемешиваем индексы примеров
    randidx = np.random.permutation(X.shape[0])

    # Берем первые K примеров как центроиды
    centroids = X[randidx[:K]]

    return centroids

## 4. Сжатие изображений с помощью K-Means
Математическая модель сжатия:

Исходное изображение:

    Размер: 128×128 пикселей

    Каждый пиксель: 3 цвета × 8 бит = 24 бита

    Общий размер: 128 × 128 × 24 = 393,216 бит

Сжатое изображение:

    Палитра: 16 цветов × 24 бита = 384 бита

    Карта пикселей: 128 × 128 × 4 бита = 65,536 бит

    Общий размер: 65,920 бит

Коэффициент сжатия: 393,216 / 65,920 ≈ 6 раз

In [ ]:
# Загрузка изображения
original_img = plt.imread('bird_small.png')

# Преобразование в матрицу RGB значений
# Исходная форма: (128, 128, 3)
# Целевая форма: (16384, 3)
X_img = np.reshape(original_img,
                   (original_img.shape[0] * original_img.shape[1], 3))

# Запуск K-means для нахождения 16 цветов
K = 16
max_iters = 10
initial_centroids = kMeans_init_centroids(X_img, K)
centroids, idx = run_kMeans(X_img, initial_centroids, max_iters)

# Восстановление сжатого изображения
# Каждому пикселю присваиваем цвет ближайшего центроида
X_recovered = centroids[idx, :]

# Возвращаем исходную форму изображения
X_recovered = np.reshape(X_recovered, original_img.shape)